<b><font size="6" color="#E8800A">Week 3b · Deepening exploration: a price, not a class</font></b><br>
<b><font size="4">The same discipline, and this file needs cleaning of its own</font></b><br>

This notebook and `week_03_deepen_exploration` are alternatives:
one target is a class, this one is a number, and the method is the same in both.
**This notebook stands alone.** It restates what it needs rather than pointing
back.

`cars4you.csv` is 4,000 used cars and the target is `price`, in euros. It is
not a file this notebook opens; it is the file this notebook **produces**, from
4,600 raw listings that carry a misspelt brand, a blank cell or a value their
own units forbid.

The defects here are not the same shape as the classification half's, so the
treatment is not the same either, and this half is where that gets settled by
measurement rather than by habit. **A treatment is not good or bad. It fits a
file or it does not**, and the only way to find out which is to run more than
one and keep the score.

The order is the same in both halves and it is the order to keep on your own
data. Explore first and change nothing, list what the exploration found, then
write the functions, then measure the alternatives through them.

<div class="alert alert-block alert-info">

# TOC<a class="anchor" id="toc"></a>
* [<font color='#E8800A'>The file, and the file it becomes</font>](#file)
* [<font color='#E8800A'>The domain rule, checked one column at a time</font>](#domain)
* [<font color='#E8800A'>Structural survey</font>](#survey)
* [<font color='#E8800A'>Distribution statistics</font>](#distributions)
* [<font color='#E8800A'>Where the boxplot fails</font>](#boxplot)
* [<font color='#E8800A'>An outlier is not always a defect</font>](#hybrids)
* [<font color='#E8800A'>Damaged copies, or different cars?</font>](#drop-not-repair)
* [<font color='#E8800A'>From exploration to a recipe</font>](#plan)
* [<font color='#E8800A'>Drop, repair or blank</font>](#treatment)
* [<font color='#E8800A'>What a missing value becomes</font>](#missing values)
* [<font color='#E8800A'>Does a transform earn its place?</font>](#transform)
* [<font color='#E8800A'>Does scaling change anything here?</font>](#scaling)
* [<font color='#E8800A'>The target itself</font>](#target)
* [<font color='#E8800A'>The recipe, the log, and the seam</font>](#recipe)
* [<font color='#E8800A'>Key takeaways</font>](#takeaways)
* [<font color='#E8800A'>References</font>](#references)

</div>

# <font color='#E8800A'>The file, and the file it becomes</font> <a class="anchor" id="file"></a>
[Back to TOC](#toc)

Two files sit in `data/raw/`. `cars4you_raw.csv` is 4,600 cars as
they were recorded; `cars4you.csv` is what a documented rule turns it into.
Every later week opens the second file and never sees the first.

That gives this notebook a property most weeks do not have: it can be checked
exactly. The rule below is applied once, here, and the last section of this
half compares what it produced against `cars4you.csv` row for row, so a wrong
decision does not start an argument; it prints `False`.

| column | what it holds | role |
|---|---|---|
| `price` | the asking price in EUR | **target** |
| `Brand` | manufacturer (9 levels) | categorical |
| `model` | model name (308 levels) | categorical, and the survey below is about it |
| `transmission` | manual, automatic, semi-auto, other (4 levels) | categorical |
| `fuelType` | petrol, diesel, hybrid, electric, other (5 levels) | categorical |
| `year` | registration year | numeric, but a year rather than a quantity |
| `mileage` | distance covered | numeric |
| `tax` | annual road tax in EUR | numeric |
| `mpg` | miles per gallon, the maker's figure | numeric |
| `engineSize` | displacement in litres | numeric |
| `paintQuality%` | condition of the paintwork, 0 to 100 | numeric |
| `previousOwners` | how many owners before this sale | numeric count |

`cars4you_raw.csv` carries one column the finished file does not: `CarID`. It
records nothing about the car. It exists so a cleaning rule can put the rows it
keeps back in the exact order `cars4you.csv` already has, which matters because
every regression week after this one splits the file by row position.

In [ ]:
import sys
from collections import namedtuple
from functools import partial
from pathlib import Path

# course_helpers.py sits in this folder, next to the notebook.
if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import (
    MinMaxScaler, OneHotEncoder, RobustScaler, StandardScaler,
)

from course_helpers import PLOT_BLUE, PLOT_ORANGE, CleaningLog

RANDOM_STATE = 42
FIGSIZE = (10, 6)

__Step 1:__ Open the raw file and see what has to change.

Run this notebook from its own folder inside the course
repository: that is what makes the `data/` paths below work. If you
downloaded this file on its own from Moodle, move it into the repository before
you run it.

In [ ]:
# pandas' fast CSV parser can round the last digit of a float; round_trip
# asks for the slower parser that recovers the exact value, which matters
# because this notebook is about to write some of these numbers straight
# back out. CarID identifies the row, so it goes in the index rather than
# sitting among the columns a model could be handed by accident.
raw = pd.read_csv("../../data/raw/cars4you_raw.csv", index_col="CarID",
                   float_precision="round_trip")

print(f"{raw.shape[0]:,} rows x {raw.shape[1]} feature columns, indexed by CarID")
raw.head()

# <font color='#E8800A'>The domain rule, checked one column at a time</font> <a class="anchor" id="domain"></a>
[Back to TOC](#toc)

Nothing below is judged against `price`. Every column, the
target included, has a domain fixed by what it measures, and a value outside
that domain is a recording defect, not evidence about the car.

<div class="alert alert-block alert-info">

**The rule, stated before the measurement.** `price` is checked the same way as
any other column: this is the target's own domain, not a decision read off a statistic.

| column(s) | domain | a value outside it is |
|---|---|---|
| `Brand`, `transmission`, `fuelType` | one of a fixed set of spellings | a keystroke slip, a clipped word or stray whitespace: `OPEL`, `Ope`, `Manua` |
| `year` | 1970 to 2025, a whole number | a fractional year, or one that has not happened |
| `mileage`, `tax` | zero or more | a negative distance or a negative currency |
| `mpg`, `engineSize` | strictly positive | a car that runs on nothing, or displaces nothing |
| `paintQuality%` | 0 to 100 | a percentage outside its own scale |
| `previousOwners` | zero or more, a whole number | a fractional or negative owner |
| `price` | strictly positive | a free car, or a negative one |

`model` has no rule of its own below: every one of its 9 missing values also
breaks another column's rule, so nothing is lost by leaving it out.

**Dropping the row is the rule here, not the last resort.**

</div>

__Step 2:__ Write the check once, so the same function serves every column.

In [ ]:
canonical = {
    "Brand": {"Audi", "BMW", "Ford", "Hyundai", "Mercedes", "Opel", "Skoda", "Toyota", "VW"},
    "transmission": {"Manual", "Automatic", "Semi-Auto", "Other", "Unknown"},
    "fuelType": {"Petrol", "Diesel", "Hybrid", "Electric", "Other"},
}


def out_of_domain(frame, cols, errors):
    """True for a row where any of `cols` fails its own rule in `errors`.

    This asks a question and answers it. It does not repair anything, and that
    separation is deliberate: which rows are wrong is settled by knowing what
    the columns mean, while what to do about them is settled further down, by
    measurement. `errors` maps each column to a predicate returning True for a
    BAD present value. A missing value is unknown rather than invalid, so it stays False
    and survives for a training fold to fill.
    """
    bad = pd.Series(False, index=frame.index)
    for column in cols:
        bad |= errors[column](frame[column])
    return bad


domain_errors = {
    "Brand": lambda values: values.notna() & ~values.isin(canonical["Brand"]),
    "transmission": lambda values: values.notna() & ~values.isin(canonical["transmission"]),
    "fuelType": lambda values: values.notna() & ~values.isin(canonical["fuelType"]),
    "year": lambda values: values.notna() & (
        ~values.between(1970, 2025) | (values != values.round())),
    "mileage": lambda values: values.notna() & (values < 0),
    "tax": lambda values: values.notna() & (values < 0),
    "mpg": lambda values: values.notna() & (values <= 0),
    "engineSize": lambda values: values.notna() & (values <= 0),
    "paintQuality%": lambda values: values.notna() & ~values.between(0, 100),
    "previousOwners": lambda values: values.notna() & (
        (values < 0) | (values != values.round())),
    "price": lambda values: values.notna() & (values <= 0),
}

bad = out_of_domain(raw, list(domain_errors), domain_errors)
print(f"{int(bad.sum()):,} of {len(raw):,} rows fail at least one column's"
      f" own rule, leaving {int((~bad).sum()):,}")

# A frame to LOOK at, so the statistics below describe cars rather than cars
# plus 600 rows that are not cars. Setting these rows aside in order to see
# past them is not the same as deciding to delete them: that decision is
# measured further down, and one of the arms there keeps 206 of them.
candidate = raw[~bad].sort_index()

**4,600 rows in, 600 fail their own column's rule, 4,000
survive.** That is the whole of the recipe's row count. The shape of the failure decides what happens to it.

In [ ]:
per_column = pd.Series({column: int(rule(raw[column]).sum())
                        for column, rule in domain_errors.items()}
                       ).sort_values(ascending=False)
print(per_column.to_string())

text_columns = ["Brand", "transmission", "fuelType"]
numeric_columns = [c for c in domain_errors if c not in text_columns]

# The same function, asked a narrower question: hand it a subset of the columns
# and it flags the rows failing those rules alone. Splitting the rule by group
# is a different question, not a reason to write the loop out a second time.
text_bad = out_of_domain(raw, text_columns, domain_errors)
num_bad = out_of_domain(raw, numeric_columns, domain_errors)

print(f"\ntext defect only     {int((text_bad & ~num_bad).sum()):4d}")
print(f"numeric defect only  {int((num_bad & ~text_bad).sum()):4d}")
print(f"both at once         {int((text_bad & num_bad).sum()):4d}")

# Missing values are counted apart from the defects, and the reason is the rule itself:
# every predicate above is guarded by `notna()`, so no row is ever flagged for
# being incomplete. A missing value is unknown, not wrong. What becomes of the ones that
# survive is decided further down, by measurement, like everything else here.
incomplete_rows = raw.isna().any(axis=1)
print(f"\nrows carrying a missing value somewhere   {int(incomplete_rows.sum()):,}")
print(f"of those, also flagged as wrong  {int((incomplete_rows & bad).sum()):,}")

<div class="alert alert-block alert-warning">

**Almost every failure is a misspelt label.** `transmission`
fails on 221 rows, `fuelType` on 179 and `Brand` on 171: a real spelling, just
not one of the canonical five, five or nine. Against that, the eight numeric
columns fail on 88 rows between them, and `price`, the target, on none at all.
`year` is the worst of them at 22.

**And a row rarely fails twice.** 514 carry only a label defect, 69 only a
numeric one, and just 17 manage both.

**A missing value is not on that list, and that is the rule working, not a
hole in the rule.** 1,469 of the 4,600 rows carry a missing cell somewhere, and only 148 of
them are flagged, and those 148 for a defect they carry as well, never for
the missing value. An empty cell says the value was not recorded; it does not say the car is
wrong. What becomes of the ones that survive into the finished file is a
question this notebook answers later, by measuring, and not by deletion.

</div>

In [ ]:
for column in text_columns:
    off_forms = sorted(raw.loc[domain_errors[column](raw[column]), column].dropna().unique())
    print(f"{column:14s} {len(off_forms)} off-canon forms, e.g. {off_forms[:6]}")

# <font color='#E8800A'>Structural survey</font> <a class="anchor" id="survey"></a>
[Back to TOC](#toc)

The domain rule has said which rows fail it. That is not the
same as knowing what the file is made of.

__Step 3:__ Look at the target itself, before looking at anything against it.
Every choice further down is judged by an error in euros, and what counts as a
large error depends on what the prices are.

In [ ]:
print(candidate["price"].describe().round(2).to_string())
print(f"\nskew {candidate['price'].skew():.4f}")

# Predicting the mean for every car is the floor any model has to clear.
floor = (candidate["price"] - candidate["price"].mean()).abs().mean()
print(f"always predicting the mean price gives a mean absolute error"
      f" of {floor:,.0f} EUR")

**Prices run from 650 to 145,000 EUR around a median of 14,396, and the
distribution leans right, skew 2.5450.** A mean absolute error is in euros, so it is read against that
spread rather than against zero: predicting the mean for every car gives
**7,145 EUR**, and a model has to beat that before its number means anything.
And a right lean in the target is the condition under which transforming the
target is worth asking about, which the last section of this notebook does.

Nothing here looks at a feature. Which columns predict the price is a question
for the modelling weeks, and asking it now, of the whole file, would be choosing
features by looking at the answers.

__Step 4:__ Confirm what the rule left, and sort its columns into numbers
and labels.

In [ ]:
print(f"{candidate.shape[0]:,} rows x {candidate.shape[1]} columns")

# Repetition and missing values, confirmed rather than assumed: every statistic after
# this one is a count over rows, so either one would change what those counts
# mean.
print("exact duplicate rows:", candidate.duplicated().sum())
print("cells missing:", int(candidate.isna().sum().sum()))

target = "price"
numeric = [c for c in candidate.columns
           if c != target and pd.api.types.is_numeric_dtype(candidate[c])]
categorical = [c for c in candidate.columns if c not in numeric + [target]]
print(f"\n{len(numeric)} numeric, {len(categorical)} categorical, target {target!r}")

levels = candidate[categorical].nunique().sort_values(ascending=False)
print("\nlevels per categorical column:\n", levels.to_string())
print(f"\nencoding all four costs {int(levels.sum())} columns;"
      f" without 'model' it costs {int(levels.drop('model').sum())}")

**No repetition, 1,537 missing values, and one column that cannot be
used as it stands.**

No duplicate rows, so every count below is a count of distinct cars. The missing values
are a different matter: the domain rule never flagged a row for carrying one, so
1,537 empty cells survive into the frame every section after this one measures.
They are left open on purpose, and the section that decides what they become
comes after the exploration that says which columns they fall in.

**`model` holds 308 levels.** Encoding the four label columns costs **326**
columns; dropping `model` first costs **18**. A 310-column block on 4,000 rows
gives many models only a handful of cars each, so their coefficients are fitted
on almost no evidence, and any model name absent from the training half has no
column to land in at all. `model` is dropped for that reason, which is a measured
one: `Brand` already carries most of what it would have said.

`year` is worth a second look too. It is stored as a number and arithmetic on it
runs, but it is a **label for when the car was registered**, and the difference
between 2016 and 2019 is not three of anything the price depends on linearly.

<div class="alert alert-block alert-info">

**The rule, stated before any of it is measured.**

A transform is **selected per column, by a paired comparison on held-out rows**,
never by a blanket rule applied to whatever looks skewed.

The reasoning: `log1p` compresses a long right tail, so it helps a column whose
tail is doing damage. Whether a particular column's tail is doing damage is a
question about that column and that model together, and skew alone cannot answer
it. The measurement below is what decides, and it is set up so that the answer
can come back **no**.

</div>

# <font color='#E8800A'>Distribution statistics</font> <a class="anchor" id="distributions"></a>
[Back to TOC](#toc)

The statistical half of the exploration. It decides which summary
the sections after it are allowed to use.

__Step 5:__ Describe the seven numeric columns.

In [ ]:
# .describe() returns a DataFrame with one COLUMN per input column.
# Transpose it and each row is a column of the file.
summary = candidate[numeric].describe().T
print(summary.round(2).to_string())

`describe()` reports count, mean, std, min, the quartiles and max.
Two things it does not report matter here: how **skewed** each column is, and
which way. What comes back is an ordinary DataFrame, so they can be added to it
as columns.

__Step 6:__ Add skew and the zero share to that same frame.

In [ ]:
shape = summary.copy()
shape["skew"] = candidate[numeric].skew()
shape["% zeros"] = 100 * (candidate[numeric] == 0).mean()
shape = shape.sort_values("skew", ascending=False)

print(shape[["mean", "50%", "max", "skew", "% zeros"]].round(2).to_string())
print(f"\nright-skewed (skew > 1): {[c for c in numeric if candidate[c].skew() > 1]}")
print(f"left-skewed  (skew < -1): {[c for c in numeric if candidate[c].skew() < -1]}")

**Three columns lean right and one leans left**, and that
distinction decides what a transform can even be asked to do.

`mpg` is the most skewed at **3.07**, then `mileage` at **1.61** and
`engineSize` at **1.33**. Those are the long-right-tail shape `log1p` is built
for.

`year` is skewed **-1.90**: its tail runs to the *left*, towards a handful of old
cars, and `log1p` does nothing about a left tail.

`previousOwners` is **19.70%** zeros and `tax` **5.68%**. A zero share that size
is not a defect here: a car with no previous owner and a car with no road tax are
both ordinary.

# <font color='#E8800A'>Where the boxplot fails</font> <a class="anchor" id="boxplot"></a>
[Back to TOC](#toc)

**What a boxplot is for.** It draws five numbers: the median, the
first and third quartiles as the box, and whiskers reaching to the furthest
points still within **1.5 x IQR** of the box. Anything past a whisker is drawn as
a dot and conventionally called an outlier. It is distribution-free, so one huge
value cannot drag the box the way it drags a mean.

The assumption inside it is that the box has width in the first place.

Two figures follow, one column each. The **blue** figure is the column the rule
suits and the **orange** figure is the column it does not, so the colour marks
which case you are looking at rather than which panel.

In [ ]:
# First the rule on a column it suits. `mileage` has a middle, and the dots
# past the whisker are a genuine handful of very high-mileage cars.
fig, (left, right) = plt.subplots(1, 2, figsize=FIGSIZE)
sns.histplot(candidate["mileage"], bins=60, color=PLOT_BLUE, ax=left)
left.set(xlabel="mileage", ylabel="cars", title="a column with a middle")
sns.boxplot(x=candidate["mileage"], color=PLOT_BLUE, ax=right)
right.set(xlabel="mileage", title="and the boxplot agrees")
plt.tight_layout()
plt.show()

# Now `tax`, where half the cars share one value. Both panels of this figure are
# orange, and both panels of the one above are blue: the colour marks the CASE,
# the column the rule suits against the column it does not, so one column is
# never drawn in two colours.
fig, (left, right) = plt.subplots(1, 2, figsize=FIGSIZE)
sns.histplot(candidate["tax"], bins=60, color=PLOT_ORANGE, ax=left)
left.set(xlabel="tax (EUR)", ylabel="cars", title="what the column is")
sns.boxplot(x=candidate["tax"], color=PLOT_ORANGE, ax=right)
right.set(xlabel="tax (EUR)", title="what the boxplot says it is")
plt.tight_layout()
plt.show()

q1, q3 = candidate["tax"].quantile([0.25, 0.75])
print(f"tax: Q1 {q1:.0f}, Q3 {q3:.0f}, IQR {q3 - q1:.0f}")
print(f"the single most common value is {candidate['tax'].mode()[0]:.0f},"
      f" on {100 * (candidate['tax'] == candidate['tax'].mode()[0]).mean():.1f}% of cars")

__Step 7:__ Apply the 1.5 x IQR rule across all seven numeric columns.

In [ ]:
def iqr_fences(frame, columns, k=1.5):
    """Lower and upper fence per column, k IQRs beyond the quartiles.

    The columns and k are arguments, so the same function answers for 1.5 and
    for any other multiplier without being rewritten.
    """
    q1, q3 = frame[columns].quantile(0.25), frame[columns].quantile(0.75)
    iqr = q3 - q1
    return q1 - k * iqr, q3 + k * iqr


def outside_fences(frame, columns, fences, **limits):
    """True for rows with a named column beyond the fences `fences` draws.

    The fence rule is an argument, so the same detector answers for the IQR rule
    here and for a standard-deviation rule elsewhere.

    A missing value counts as INSIDE. It has to: a comparison against a missing value is
    False both ways, so without the `isna()` term below, every row carrying a missing value
    would be called an outlier and the rule would look far more aggressive than
    it is.
    """
    low, high = fences(frame, columns, **limits)
    within = frame[columns].ge(low) & frame[columns].le(high)
    return ~(within | frame[columns].isna()).all(axis=1)


# The mask comes from the detector. The fences are read off separately because
# the per-column report below wants the numbers themselves, not the mask.
low, high = iqr_fences(candidate, numeric, k=1.5)
kept = ~outside_fences(candidate, numeric, iqr_fences, k=1.5)
print(f"rows kept {kept.sum():,} of {len(candidate):,}"
      f"  ({100 * (1 - kept.mean()):.2f}% removed)")
print("\nrows each column flags on its own:")
# `between` is False for a missing value, so `~between` would call every missing value an
# outlier, which is the mistake `outside_fences` carries an `isna()` term to
# avoid. The report has to make the same allowance the detector does.
print({c: int((~candidate[c].between(low[c], high[c])
               & candidate[c].notna()).sum()) for c in numeric})
print(f"\nmean price kept    {candidate.loc[kept, target].mean():,.0f} EUR")
print(f"mean price deleted {candidate.loc[~kept, target].mean():,.0f} EUR")

<div class="alert alert-block alert-warning">

**The rule removes 29.43% of the file, and 1,043 of those rows are
flagged by `tax` alone.**

`tax` has Q1 **125** and Q3 **145**, because **44.8%** of the cars sit at exactly
145. An interquartile box 20 euros wide puts the fences at 95 and 175, so every
tax-free electric car and every high-tax large engine falls outside. Nothing is
wrong with the arithmetic. The rule assumes a distribution with a middle, and a
column where nearly half the rows share one value does not have one.

**And what it removes is not a random third.** The mean price of the rows it
keeps is **18,442 EUR**; of the rows it deletes, **12,617 EUR**. Cheap cars are
old, high-mileage and tax-banded away from 145, so a rule that deletes extreme
values deletes the cheap end of the market. Fit a price model on what survives
and it has never seen the cars it will most often be asked about.

`paintQuality%` and `previousOwners` flag **nothing at all**. Stacking seven
tests with an *and* means the two well-behaved columns cannot rescue a row that
any of the other five rejects.

</div>

# <font color='#E8800A'>An outlier is not always a defect</font> <a class="anchor" id="hybrids"></a>
[Back to TOC](#toc)

The largest `mpg` in the file is **235.4**, against a median of
54.3. A rule that reads one column at a time calls that a data-entry error.

__Step 8:__ Ask what those cars are, before deciding they are wrong.

In [ ]:
extreme = candidate[candidate["mpg"] > 100]
print(f"{len(extreme)} cars report more than 100 mpg\n")
print(extreme["fuelType"].value_counts().to_string())
print("\nmedian mpg by fuel type:")
print(candidate.groupby("fuelType")["mpg"].median().sort_values(
    ascending=False).round(1).to_string())

<div class="alert alert-block alert-warning">

**They are hybrids, and the 235.4 is the electric car.**

Of the 29 cars above 100 mpg, **21 are hybrids** and one is the file's only
electric. Read per fuel type the numbers are unremarkable: the median is
**235.4** for electric, **78.0** for hybrid and **51.4** for petrol. These rows
are not corrupt; they are a different kind of car, and `fuelType` says so in the
next column along.

**This is the limit of the section above.** A fence drawn on the marginal
distribution of one column cannot see a second column that explains it.
Deleting these rows would not remove noise, it would remove the hybrids, and a
price model that has never seen a hybrid is not better for it.

So the 1.5 x IQR rule is a **screening device**, not a verdict. What a flagged row turns out to be is a question about the data,
answered by looking at it.

</div>

# <font color='#E8800A'>Damaged copies, or different cars?</font> <a class="anchor" id="drop-not-repair"></a>
[Back to TOC](#toc)

A misspelt `Brand` could mean two quite different things. It could
be a good record with one cell typed badly, in which case repairing the cell
recovers a car. Or it could be a car that was never in this dataset, carrying a
spelling nobody standardised because nobody was meant to read it. The treatment
turns on which, and that is a question about the rows rather than about the
spelling.

__Step 9:__ Check whether a flagged row is a damaged copy of a kept one, or a different car.

In [ ]:
match_columns = ["price", "mileage", "year", "engineSize"]
dropped = raw.loc[bad, match_columns].apply(tuple, axis=1)
kept_keys = set(raw.loc[~bad, match_columns].apply(tuple, axis=1))
shares_a_kept_row = dropped.isin(kept_keys)
print(f"{int(shares_a_kept_row.sum())} of the {len(dropped):,} dropped rows share"
      f" {', '.join(match_columns)} with a row the rule kept")

<div class="alert alert-block alert-warning">

**3 of 600.** Almost none of the flagged rows are damaged copies
of a row the rule kept: on `price`, `mileage`, `year` and `engineSize` together,
597 of them match nothing that survived. They are different cars, not damaged
records of the same cars, so repairing a spelling here would not recover
anything. It would add a car that was not in the dataset.

That is an argument, and the next section turns it into a measurement, because
an argument about what rows *are* can be right while still being wrong about
what to *do* with them. **514 of the 600 carry nothing but a spelling defect.**
Their price, mileage and every other cell are ordinary, and a repair would hand
the model 514 more cars to learn from.

</div>

# <font color='#E8800A'>From exploration to a recipe</font> <a class="anchor" id="plan"></a>
[Back to TOC](#toc)

The exploration is over and nothing has been written. The domain
rule was stated before any of it, from what the columns mean, and everything
since has been a way of finding out what the file does with that rule.

<div class="alert alert-block alert-info">

**What the exploration found, and what is still undecided.**

| # | what the exploration found | what is still open |
|---|---|---|
| 1 | The domain rule flags 600 rows of 4,600. 514 carry a label defect and nothing else, 69 a numeric one and nothing else, and 17 fail both at once. | What happens to a flagged row: drop it, repair the spelling, or blank the offending cell. |
| 2 | 597 of the 600 match no surviving row on `price`, `mileage`, `year` and `engineSize`, so they are different cars rather than damaged copies. | Nothing about what they ARE. Everything about whether keeping the repairable 514 helps a model. |
| 3 | The rule flags no row for being incomplete, so 1,537 missing values survive into the finished file. | What a missing value becomes: which value fills it, whether a label missing value is filled at all, and where that value may be computed. |
| 4 | Three numeric columns lean right and `year` leans left. | Whether a transform earns its place, column by column. |
| 5 | `price` is skewed on the training rows. | Whether the target itself should be transformed. |
| 6 | The 1.5 x IQR rule deletes cars that are expensive rather than wrong, and the 29 cars above 100 mpg are hybrids and one electric. | Nothing. An outlier that a domain explanation accounts for is not a defect, and the rule that would delete it has already failed. |

One of the six is closed by the exploration itself, and the rest are measured
below. Nothing is written to disk until every one of them has an answer.

</div>

__Step 10:__ Write the function that carries out a decision about a flagged
row, and the three fixes it can be given. The rule and the treatment were kept
apart on purpose: `out_of_domain` above says which rows are wrong, and this says
what happens to them.

**Everything that changes the data is here, and nothing else is.** A function
that measures belongs beside the measurement it serves, which is why the two
scorers further down are written where they are used rather than here.

**Both functions take their treatment as an argument**, which is what lets the
alternatives be compared through the same code that will carry out the winner.
`apply_domain_rule` decides about a whole row; `fill_missing` decides what an
empty cell becomes.

In [ ]:
def apply_domain_rule(frame, cols, errors, fix):
    """Flag rows failing their own column's rule, then hand them to `fix`.

    No column, rule or repair is fixed here: all three are arguments, so the
    same function serves a recipe that drops and a recipe that repairs, and the
    two can be compared without a second function that might drift from this
    one. Every treatment is handed the same two things, `frame` and `bad`, which
    is what makes them interchangeable here; only dropping reads `bad`, because
    only dropping acts on whole rows.
    """
    bad = out_of_domain(frame, cols, errors)
    return fix(frame, bad)


def drop_flagged(frame, bad):
    """Drop it: the flagged rows leave, and nothing is guessed."""
    return frame.loc[~bad]


def nearest_canon(value, canon):
    """The canonical form `value` matches, or `value` unchanged.

    The matching rule is deliberately crude, because a crude rule you can read
    beats a clever one you cannot: a value matches a canonical form when either
    contains the other, once case and surrounding space are removed. That is
    enough for `BM`, `Aud`, `HYUNDA`, `Manua`, `etrol` and `iesel`, and it
    leaves anything it cannot match alone, where the rule will still flag it.
    """
    if pd.isna(value):
        return value
    probe = str(value).strip().lower()
    for good in sorted(canon):
        if probe in good.lower() or good.lower() in probe:
            return good
    return value


def repair_to_canon(frame, bad, allowed, cols):
    """Repair it: map each off-canon spelling to the canonical form it contains.

    `allowed` and `cols` are arguments, so the same repair serves any label
    column against any canon.
    """
    out = frame.copy()
    for column in cols:
        out[column] = out[column].map(
            partial(nearest_canon, canon=allowed[column]))
    return out


def blank_flagged(frame, bad, cols, errors, replacement=np.nan):
    """Blank it: the offending cells become `replacement` and the row stays.

    `bad` says which ROW failed and blanking has to know which CELL did, so
    this one asks the rules again, column by column. `replacement` is an
    argument for the reason it is one in the classification half: a blank
    records that the value is not known, while a 0 would claim it was meant to
    be zero, and a claim can be wrong.
    """
    out = frame.copy()
    for column in cols:
        out.loc[errors[column](out[column]), column] = replacement
    return out


def fill_missing(train, test, plan):
    """Fill every missing value with a value computed from `train` alone.

    `plan` is a list of (columns, fill) pairs applied in order, and `fill` takes
    a column and returns the value its missing values become. A column named in no pair
    keeps its missing values, so a column nobody decided about stays visible instead of
    being filled by a default. A label column left out this way stays available
    for the encoder to give a level of its own.

    One fill for the whole frame could not give the numeric columns a median and
    the label columns a level, which is the first thing this file needs.

    Every value comes from `train` and is then applied to both halves, and that
    is the whole guard. A median computed over the two halves together would
    have been computed partly from the rows the model is about to be scored on.
    """
    train, test = train.copy(), test.copy()
    for columns, fill in plan:
        for column in columns:
            value = fill(train[column])
            train[column] = train[column].fillna(value)
            test[column] = test[column].fillna(value)
    return train, test

# <font color='#E8800A'>Drop, repair or blank</font> <a class="anchor" id="treatment"></a>
[Back to TOC](#toc)

Three treatments, one function, and a comparison that has to be set
up carefully: the arms produce frames of different lengths, so left alone each
one would be examined on a test set of its own choosing. **One set of test cars,
fixed in advance, and each arm may only differ in what it trains on.**

__Step 11:__ Apply each treatment through `apply_domain_rule`, and see what each
one leaves behind before scoring any of them.

In [ ]:
treatments = {
    "drop the row": drop_flagged,
    "repair the spelling": partial(repair_to_canon, allowed=canonical,
                                   cols=text_columns),
    "blank the cell": partial(blank_flagged, cols=list(domain_errors),
                              errors=domain_errors),
}
treated = {}
for name, fix in treatments.items():
    frame = apply_domain_rule(raw, list(domain_errors), domain_errors, fix)
    if name == "blank the cell":
        # A blanked cell is missing, and a missing value fails every rule, so
        # re-applying the rule to this arm would empty the frame.
        treated[name] = frame
    else:
        # A repair fixes spellings but cannot invent a price, so whatever still
        # fails after the treatment still has to go. That is what "repair what
        # you can, drop what you cannot" means when it is carried out.
        still_bad = out_of_domain(frame, list(domain_errors), domain_errors)
        treated[name] = frame.loc[~still_bad]

print(pd.DataFrame.from_dict(
    {name: {"rows": len(frame),
            "missing values": int(frame.isna().sum().sum()),
            "median price": frame["price"].median()}
     for name, frame in treated.items()},
    orient="index").to_string())

__Step 12:__ Score the three on the same test cars, over ten draws. Each arm
trains on its own surviving rows minus every car in that draw's test set, so
the only thing that varies between the arms is what they were allowed to learn
from.

In [ ]:
# Every arm arrives carrying missing values, because the file ships them open and the
# blank arm adds more of its own, and a linear model cannot fit a missing value. So each
# arm fills inside its own training rows, and the numbers it fills with are
# computed from those rows alone. Filling with zero would answer "is blanking
# worse than a bad fill", which is not the question this table asks.
def sem(values):
    """Standard error of the mean: how far it would move on new folds."""
    return values.std(ddof=1) / np.sqrt(len(values))


def encoded(half, encoder, numbers, labels):
    """The numeric columns beside the encoder's one-hot columns, in one frame.

    The encoder arrives already fitted, on training rows, and this is the only
    place either half is turned into a design matrix -- so a half can never be
    encoded against a vocabulary the other half's rows helped choose.
    """
    hot = pd.DataFrame(encoder.transform(half[labels]),
                       columns=encoder.get_feature_names_out(labels),
                       index=half.index)
    return pd.concat([half[numbers], hot], axis=1)


def mae_training_on(frame, holdout):
    """Fit on this arm's rows, predict the test cars the drop arm supplied."""
    test = treated["drop the row"].loc[holdout].drop(columns=["model"])
    train = frame.drop(index=holdout, errors="ignore").drop(columns=["model"])
    features = [c for c in train.columns if c != target]
    labels = [c for c in features if not pd.api.types.is_numeric_dtype(train[c])]
    numbers = [c for c in features if c not in labels]

    train, test = fill_missing(train, test,
                            [(numbers, lambda column: column.median()),
                             (labels, lambda column: "(missing)")])
    encoder = OneHotEncoder(drop="first", sparse_output=False, dtype=float,
                            handle_unknown="ignore").fit(train[labels])

    model = LinearRegression().fit(encoded(train, encoder, numbers, labels),
                                   train[target])
    return mean_absolute_error(
        test[target],
        model.predict(encoded(test, encoder, numbers, labels)))


# Ten draws, not one. A single 800-car test set moves these numbers by more
# than the arms differ by, so one draw would report the split rather than the arm.
holdouts = [treated["drop the row"].sample(frac=0.2, random_state=seed).index
            for seed in range(10)]
arm_scores = {name: np.array([mae_training_on(frame, h) for h in holdouts])
              for name, frame in treated.items()}
control = arm_scores["drop the row"]

per_arm = {}
for name, scores in arm_scores.items():
    difference = scores - control
    per_arm[name] = {
        "training rows": len(treated[name].drop(index=holdouts[0], errors="ignore")),
        "MAE": scores.mean(),
        "vs drop": difference.mean(),
        "paired SEM": sem(difference),
        "beats drop on": f"{int((difference < 0).sum())} of {len(difference)}",
    }
print(pd.DataFrame.from_dict(per_arm, orient="index").round(2).to_string())

<div class="alert alert-block alert-warning">

**The measurement does not agree with the file, and saying so is
the point of having measured.**

`repair the spelling` beats `drop the row` by **12.82 EUR, with a paired
standard error of 2.64, on 10 of the 10 draws.** That is nearly five standard
errors: as clean a result as anything in this notebook. It comes from 514 extra
training cars, and it is **0.41%** of a 3,139.51 EUR error.

`blank the cell` lands **0.45 EUR from dropping, with a paired SEM of 2.64**,
and is worse on five of the ten draws. The uncertainty is several times the
difference, so what the table supports is that blanking and dropping are the same
on this file, and blanking gets there while training on 600 more rows than
dropping does. It buys nothing here. **A
treatment is not good or bad; it fits a file or it does not**, and this file has
a treatment that beats both.

**So why does `cars4you.csv` carry 4,000 rows and not 4,514?** Not because
repair lost. Because this file is a fixed teaching artifact: eleven later
notebooks open it and split it by position with a fixed seed, so a file of a
different length would move every published number in the rest of the course.
That is a real constraint, but notice what kind of reason it is. It is a reason about this course, not about these cars. On your own
project there is no such constraint, and on the evidence above **you should
repair the 514 rows**.

The habit to take from this is not the rule. It is having a number to put beside
the rule, so that when a constraint overrides the measurement you can say by how
much it cost: **12.82 EUR of mean absolute error, or under half of one
percent.**

</div>

# <font color='#E8800A'>What a missing value becomes</font> <a class="anchor" id="missing values"></a>
[Back to TOC](#toc)

The row treatment is settled. The cells that are still empty
are not: `cars4you.csv` ships with its missing values open, on purpose, so that the week
which opens it decides what they become instead of inheriting a decision made
once by whoever wrote the file.

A fill is a value invented and put where a measurement should have been, so it
has to be **fitted on the training rows and applied to both halves**. Fitted on
the whole file it would carry the test cars' own answers back into the rows the
model trains on.

__Step 13:__ Build the scorer both comparisons run on.

In [ ]:
def car_fold(frame, numeric, categorical, target, plan, log_columns, seed,
             drop=("model",), test_size=0.2, scaler=None):
    """Everything that happens to one 80/20 split, in the order it happens.

    The dataset, the column roles, the target, the fill plan, the columns to
    log, the seed, the scaler, the columns to discard and the size of the test
    half are all arguments, so one function serves every comparison below
    rather than a copy of this loop per question.

    **The split comes first, and everything else is fitted after it.** The fill
    reads only the training rows, the encoder learns its categories from only
    the training rows, and the test rows are touched once, to be scored on.
    Fitting any of the three on the whole frame would let the test cars inform
    the numbers the model was built with.

    Returns the two design matrices and the two targets, in that order.
    """
    frame = frame.drop(columns=list(drop))
    numeric = [c for c in numeric if c in frame.columns]
    categorical = [c for c in categorical if c in frame.columns]

    train, test = train_test_split(frame, test_size=test_size,
                                   random_state=seed)
    train, test = fill_missing(train, test, plan)
    for half in (train, test):
        half[log_columns] = np.log1p(half[log_columns].clip(lower=0))

    # handle_unknown="ignore" because a level holding one car sometimes
    # lands entirely in the test half. A fold that crashed for that reason
    # would be reporting on the split rather than on the thing under test.
    encoder = OneHotEncoder(drop="first", sparse_output=False, dtype=float,
                            handle_unknown="ignore").fit(train[categorical])

    train_X = encoded(train, encoder, numeric, categorical)
    test_X = encoded(test, encoder, numeric, categorical)
    if scaler is not None:
        # Fitted on the training half, like every other fitted thing here.
        # A scaler fitted on the whole frame would carry the test cars'
        # spread back into the rows the model trains on.
        fitted = scaler.fit(train_X)
        train_X = pd.DataFrame(fitted.transform(train_X),
                               columns=train_X.columns, index=train_X.index)
        test_X = pd.DataFrame(fitted.transform(test_X),
                              columns=test_X.columns, index=test_X.index)
    return train_X, test_X, train[target], test[target]


CarScore = namedtuple("CarScore", "mae train_mae rmse train_rmse")


def car_score(frame, numeric, categorical, target, plan, log_columns, seed,
              drop=("model",), test_size=0.2, scaler=None):
    """Fit on one fold's training rows and score on the rows it did not see.

    Returns a `CarScore`: mean absolute error on the held-out cars under
    `.mae`, the same on the cars the model was fitted on under `.train_mae`,
    and root mean squared error on both sides under `.rmse` and `.train_rmse`.

    **The training scores are here so a board can show what a held-out column
    alone cannot**: a candidate scoring far better on the cars it fitted than on
    the cars it did not is memorising them, and a held-out number by itself
    cannot tell you that.

    **Two metrics rather than one**, and both in euros, so they sit side by
    side. They answer different questions. MAE is the average euro you are
    wrong by, and every car counts the same. RMSE squares the error before
    averaging, so a handful of badly wrong cars move it while barely moving
    MAE. Where the two agree the reading is firmer than either alone, and where
    they disagree the disagreement is itself the finding.

    It takes the same arguments as `car_fold` and hands them straight on, so
    the board below can ask for a score and the scaling section can ask for the
    matrix that score was computed on, without either one holding a copy of the
    other's loop.
    """
    train_X, test_X, train_y, test_y = car_fold(
        frame, numeric, categorical, target, plan, log_columns, seed,
        drop=drop, test_size=test_size, scaler=scaler)
    model = LinearRegression().fit(train_X, train_y)
    fitted_cars, held_out = model.predict(train_X), model.predict(test_X)
    return CarScore(mean_absolute_error(test_y, held_out),
                    mean_absolute_error(train_y, fitted_cars),
                    root_mean_squared_error(test_y, held_out),
                    root_mean_squared_error(train_y, fitted_cars))

__Step 14:__ Name the fills, then score six plans on the same ten folds. Two
of them differ only in what a categorical missing value becomes, so their paired row
measures that choice alone.

In [ ]:
# `model` is dropped before the model matrix is built, so the plans
# name the three label columns that survive rather than all four.
labels = [c for c in categorical if c != "model"]

# A fill takes a column and returns the value its missing values become. Written this way
# they are values, so a plan is data rather than a branch, and adding a seventh
# candidate below is one more line rather than another `if`.
fills = {"median": lambda column: column.median(),
         "mean": lambda column: column.mean(),
         "zero": lambda column: 0,
         "mode": lambda column: column.mode().iloc[0],
         "own level": lambda column: "(missing)"}

# Which numeric columns a mean would misrepresent, by the skew the distributions
# section already measured rather than by re-deriving it from the dtype.
column_skew = candidate[numeric].skew()
skewed = column_skew[column_skew.abs() > 1].index.tolist()
symmetric = column_skew[column_skew.abs() <= 1].index.tolist()

print(f"{len(labels)} label columns to decide about: {labels}")
print(f"skewed enough that a mean would misrepresent them: {skewed}")

In [ ]:
plans = {
    "median / own level": [(numeric, fills["median"]),
                           (labels, fills["own level"])],
    "mean / own level": [(numeric, fills["mean"]), (labels, fills["own level"])],
    "zero / own level": [(numeric, fills["zero"]), (labels, fills["own level"])],
    "skew-aware / own level": [(skewed, fills["median"]),
                               (symmetric, fills["mean"]),
                               (labels, fills["own level"])],
    "median / training mode": [(numeric, fills["median"]),
                               (labels, fills["mode"])],
    # No pair names a label column here, so `fill_missing` leaves those missing values alone
    # and OneHotEncoder gives each one a column of its own.
    "median / missing left to encode": [(numeric, fills["median"])],
}

seeds = range(10)
scored = {name: [car_score(candidate, numeric, categorical, target,
                           plan, [], seed) for seed in seeds]
          for name, plan in plans.items()}


def across_folds(folds, metric):
    """One metric over the ten folds, in the order the folds were drawn."""
    return np.array([getattr(fold, metric) for fold in folds])


print(pd.DataFrame({
    name: {"MAE": across_folds(folds, "mae").mean(),
           "SEM": sem(across_folds(folds, "mae")),
           "train MAE": across_folds(folds, "train_mae").mean(),
           "RMSE": across_folds(folds, "rmse").mean(),
           "train RMSE": across_folds(folds, "train_rmse").mean()}
    for name, folds in scored.items()}).T.round(2).to_string())

reference = scored["median / own level"]
print("\npaired against median / own level, positive = worse:")
for name, folds in scored.items():
    if name == "median / own level":
        continue
    gap = across_folds(folds, "mae") - across_folds(reference, "mae")
    # The same paired difference on RMSE, because a candidate that wins on the
    # average euro and loses on the badly wrong cars has not simply won.
    rmse_gap = across_folds(folds, "rmse") - across_folds(reference, "rmse")
    print(f"  {name:28s} MAE {gap.mean():+8.2f} +/- {sem(gap):5.2f} EUR"
          f"   {abs(gap.mean()) / sem(gap):5.2f} SE,"
          f" worse on {int((gap > 0).sum())} of {len(gap)}"
          f"   RMSE {rmse_gap.mean():+8.2f}")

<div class="alert alert-block alert-info">

**Why the board carries four numbers and not one.**

**The training columns answer a question the held-out ones cannot.** A
candidate scoring far better on the cars it was fitted on than on the cars it
was not is memorising them, and a held-out number by itself never says so. Here
the winning arm sits at **3,196.57 EUR held out against 3,127.15 fitted**, a gap
of **69.42 EUR**, just over 2%, and every other arm shows about the same gap. So
nothing on this board is memorising, and the column earns its place by ruling
that out rather than by separating the candidates.

**RMSE is the second opinion.** Both metrics are in euros, so they sit side by
side, but they weigh a mistake differently: MAE is the average euro you are
wrong by and counts every car the same, while RMSE squares the error first, so a
few badly wrong cars move it and barely move MAE. That **5,190.07 against
3,196.57** is not a worse score, it is the same errors read on a scale that
notices the tail, and the gap between them says the tail exists.

**Use it to break a tie, and read a disagreement as a warning.** Where the board
moves at all the two agree: zero costs **639.03** on MAE and **808.13** on RMSE,
the training mode **9.76** and **16.18**. The one arm where they disagree is
`mean / own level`, **+0.97** on MAE and **-1.16** on RMSE, and both numbers are
smaller than the noise they were measured in. A disagreement inside the noise is
not a finding about the two metrics. It is the noise.

</div>

<div class="alert alert-block alert-info">

**What SEM is, and why every board in this notebook carries one.**

Each row of the board is ten numbers, one per fold: the same comparison run on
ten different train/test splits. The **mean** is what that candidate scored on
average, and the **standard error of the mean**, SEM, is how far that average
would move if you drew ten more folds. It is the spread of the ten divided by
the square root of ten, and it shrinks as you add folds, which is the whole
reason for running ten rather than one.

Read the two together and never the mean alone. A difference of +9.76 with a
SEM of 3.23 is about three times its own uncertainty and is a result. A
difference of +0.97 with a SEM of 1.33 is smaller than the noise it was measured
in, and it should be read as **no difference**, not as "slightly
worse".

</div>

In [ ]:
# The fill plan this board chose.
chosen_plan = plans["median / own level"]


def car_mae(log_columns, seed):
    """The held-out MAE alone, for the boards below that compare one number.

    `car_score` returns four. The transform comparison asks whether a column is
    better logged than not, and that is a question about the average euro, so
    it takes `.mae` and leaves the other three.
    """
    return car_score(candidate, numeric, categorical, target, chosen_plan,
                     log_columns, seed).mae


print(f"one 80/20 split, seed {RANDOM_STATE}, nothing transformed:"
      f" {car_mae([], RANDOM_STATE):,.2f} EUR")

# <font color='#E8800A'>Does a transform earn its place?</font> <a class="anchor" id="transform"></a>
[Back to TOC](#toc)

Three columns lean right and one leans left, which is the shape
`log1p` is for. The question is not whether they are skewed, which the table
already answered, but whether transforming them makes the predictions
better.

__Step 15:__ Score every column on its own, then the two blanket rules,
against the same ten seeds.

In [ ]:
seeds = range(10)
untouched = np.array([car_mae([], seed) for seed in seeds])

skewed = [c for c in numeric if abs(candidate[c].skew()) > 1]

# One entry per candidate: its name, and the columns it takes the log of.
log_candidates = {f"log1p {c}": [c] for c in numeric}
log_candidates["log1p every skewed column"] = skewed
log_candidates["log1p all seven"] = numeric

# The same measurement for all nine: ten seeds, minus the same ten untouched
# baselines, so every row is paired against the row above it.
verdict = {name: np.array([car_mae(columns, seed) for seed in seeds]) - untouched
           for name, columns in log_candidates.items()}

print(f"baseline MAE {untouched.mean():.2f} EUR;"
      f" negative = the transform earned its place\n")
print(pd.DataFrame({"MAE change (EUR)": {k: v.mean() for k, v in verdict.items()},
                    "SEM": {k: sem(v)
                            for k, v in verdict.items()}}).round(2).to_string())
print("\nskewed columns (|skew| > 1):", skewed)

# The board settles mpg and rejects two columns outright. It leaves two open,
# so score the two recipes those two allow, on the same ten folds.
pair = {"log1p mpg, tax": ["mpg", "tax"], "log1p mpg, mileage": ["mpg", "mileage"]}
paired = {name: np.array([car_mae(columns, seed) for seed in seeds]) - untouched
          for name, columns in pair.items()}
print("\nthe two recipes the board leaves open:\n")
print(pd.DataFrame({"MAE change (EUR)": {k: v.mean() for k, v in paired.items()},
                    "SEM": {k: sem(v)
                            for k, v in paired.items()}}).round(2).to_string())
print("\nraw range of each open column:")
print((candidate[["tax", "mileage"]].max()
       - candidate[["tax", "mileage"]].min()).round(0).to_string())
print()
print("year, before and after log1p: standard deviation"
      f" {candidate['year'].astype('Float64').std():.3f}"
      f" then {np.log1p(candidate['year'].astype('Float64')).std():.3f}")

<div class="alert alert-block alert-warning">

**A blanket rule loses to a per-column test, and the reason is
visible in the table.**

Against a baseline MAE of **3,196.57 EUR** (lower is better, so a negative change
is a gain):

| column | skew | MAE change | verdict |
|---|---|---|---|
| `mpg` | 3.07 | **-44.63 +/- 5.47** | earned |
| `tax` | -0.19 | -2.89 +/- 0.83 | real, and too small to act on |
| `mileage` | 1.61 | +6.53 +/- 14.41 | no effect either way |
| `engineSize` | 1.33 | **+111.59 +/- 6.30** | **rejected** |
| `year` | -1.90 | +0.61 +/- 0.03 | flattened, and it costs almost nothing |

**`engineSize` is the one to look at.** It is skewed, it is the strongest single
predictor of price in the file, and logging it costs **112 EUR per car**: its
values run from about 1 to 6 litres, so `log1p` compresses almost nothing and
bends a relationship that was already close to linear.

**`year` is the instructive opposite, and the reason to read the column rather
than guess at it.** Its skew is to the *left*, every value sits near 2017, and
taking logs leaves a column whose standard deviation is 0.001. That sounds like
destroying it, and it costs **0.61 EUR**. A least-squares fit is free to
multiply a flattened column by a correspondingly larger coefficient, so
flattening by itself is nearly free here. What is not free is bending a
relationship that was already straight, which is what `engineSize` does.

**So the plausible rule loses.** *"log1p every skewed column"* is **+83.35 EUR
worse** than doing nothing and *"log1p all seven"* is **+78.62 EUR worse**, both
dragged there by the one column the per-column test rejects.

`paintQuality%` and `previousOwners` are the two rows not in the table above:
-0.02 +/- 0.36 and -0.00 +/- 0.20, both far smaller than their own uncertainty,
which is what "no effect" looks like when it is measured properly.

`tax` leans very slightly left, at -0.19, and still lowers the error. Skew
was never the criterion; it was only ever a reason to look.

One more thing the seeds bought you: a single split put the untouched baseline at
**3,307.92 EUR** and ten splits average **3,196.57**. The difference between those
two is larger than seven of the nine effects in the table.

**Which is the reason this board settles one column and not three.** `mpg` earns
its transform eight times over its own uncertainty, and `engineSize` loses by
enough to be certain about. `tax` at -2.89 and `mileage` at +6.53 are different:
a 3 EUR effect and a 7 EUR effect on a 3,196 EUR baseline are both far inside the
111 EUR the choice of split moves, so the score is not entitled to decide between
them, whichever way the sign happens to point.

**The second table is the one that matters, and it costs 21.14 EUR to read it the
way this course does.** Keeping `tax` scores -47.58, taking `mileage` instead
scores -26.44, and the difference sits inside the same noise. What separates the
two is not the score but the columns: `mileage` spans about 150,000 and `tax`
spans 580, and a column two orders of magnitude wider than its neighbours governs
any distance computed before scaling and takes the first step of any gradient with
it. That is a property of the column rather than of the model that happened to
measure it, which is what makes it the one to decide on when the score cannot.

**So the recipe records `log1p` on `mpg` and `mileage`.** It is the more expensive
of the two by 21.14 EUR on this week's model, and it is the one that behaves the
same way in every week that inherits it. A recipe tuned to the model that measured
it is not a recipe; it is that model's preprocessing, wearing the file's name.

</div>

# <font color='#E8800A'>Does scaling change anything here?</font> <a class="anchor" id="scaling"></a>
[Back to TOC](#toc)

Sweeping a set of scalers and picking whichever scores best is
the obvious move here, and it is the wrong one. Make that mistake
out loud, because ordinary least squares on a full-rank design is
**scale-invariant in fit**. Multiply a column by a thousand and its
coefficient is divided by a thousand; the predictions do not move. A scaler
board on this model cannot report a better score, because there is not one to
report.

It can report something else, and the something else is what makes scaling
matter in every week after this one.

__Step 16:__ Score the four, and measure the design matrix each one hands the
solver.

In [ ]:
scalers = {"none": None, "standard": StandardScaler(),
           "min-max": MinMaxScaler(), "robust": RobustScaler()}


def conditioning(design):
    """How near the design matrix is to being singular, intercept included.

    The condition number is the ratio of the matrix's largest singular value to
    its smallest. It measures how much the solve can amplify a small change in
    the inputs, so it is a property of the MATRIX rather than of the fit, which
    is exactly why a scaler can move it while leaving the score where it was.
    """
    with_intercept = np.column_stack(
        [np.ones(len(design)), design.to_numpy(float)])
    return np.linalg.cond(with_intercept)

In [ ]:
rows = {}
for name, scaler in scalers.items():
    scorer = partial(car_score, candidate, numeric, categorical, target,
                     chosen_plan, scaler=scaler)
    folder = partial(car_fold, candidate, numeric, categorical, target,
                     chosen_plan, scaler=scaler)
    scores = np.array([scorer([], seed).mae for seed in seeds])
    # car_fold returns (train_X, test_X, train_y, test_y); the training design
    # is the matrix the solver was actually handed.
    numbers = [conditioning(folder([], seed)[0]) for seed in seeds]
    rows[name] = {"MAE": scores.mean(),
                  "SEM": sem(scores),
                  "cond(X)": np.mean(numbers)}

scaler_board = pd.DataFrame.from_dict(rows, orient="index")
# Per column, because the whole point of the table is a MAE difference in the
# second decimal place beside a condition number in the tens of millions, and
# one format cannot show both.
print(scaler_board.to_string(formatters={
    "MAE": lambda value: f"{value:,.2f}",
    "SEM": lambda value: f"{value:.2f}",
    "cond(X)": lambda value: f"{value:,.4g}"}))
print()
print(f"MAE spread across the four:"
      f" {scaler_board['MAE'].max() - scaler_board['MAE'].min():.2f} EUR,"
      f" against a SEM of about {scaler_board['SEM'].mean():.0f}")
print(f"cond(X) falls by a factor of"
      f" {scaler_board.loc['none', 'cond(X)'] / scaler_board.loc['standard', 'cond(X)']:,.0f}"
      f" from none to standard")

<div class="alert alert-block alert-warning">

**The scaler does not change the fit. It changes how hard the
fit is to compute.**

The four rows agree on MAE to within **1.32 EUR**, against a standard error of
about **40**. That is a thirtieth of one standard error, which is no difference
under any reading, and it is not luck: least squares solves for the
coefficients that minimise squared error, and rescaling a column rescales its
coefficient by exactly the reciprocal. The fitted values are the same numbers
reached by a different route.

What moves is the **condition number**, from **4.3e+07** with no scaler to
**12.75** under standard scaling, a factor of about three and a half million.
`mileage` runs to 152,420 and `engineSize` to 5.5, so the unscaled design holds
columns whose scales differ by five orders of magnitude, and the ratio of its
largest singular value to its smallest inherits that difference.

Look at the three scaled rows: they agree with each other to the second decimal
place, while the unscaled row sits 1.32 EUR away from them. That gap is not a
modelling difference between scaled and unscaled. It is the unscaled solve's own
arithmetic error, which is what a condition number of 43 million costs you
even in double precision.

**So why scale at all?** Because scale-invariance is a property of ordinary
least squares, not a property of models. A k-nearest-neighbours model measures
distances, and a column running to 152,420 drowns one running to 5.5. Ridge and
lasso penalise the size of a coefficient, so they penalise a column for the
units it happens to be recorded in. Anything fitted by gradient descent takes
its step size from the scale of the inputs.

This board is the **control case**. It shows what the answer looks like when
scaling does not matter, so that when a later week runs the same board
and the score does move, you can tell that the movement is real.

</div>

# <font color='#E8800A'>The target itself</font> <a class="anchor" id="target"></a>
[Back to TOC](#toc)

Every column above was a feature. The target has a shape too, and
on a price it is almost always the same shape.

__Step 17:__ Measure the skew of `price` on the training rows, and of its log.

In [ ]:
train_price, _ = train_test_split(
    candidate[target], test_size=0.2, random_state=RANDOM_STATE)

print(f"price      skew {train_price.skew():.4f}")
print(f"log1p      skew {np.log1p(train_price).skew():.4f}")
print(f"\nmedian {train_price.median():,.0f} EUR,"
      f" mean {train_price.mean():,.0f} EUR,"
      f" max {train_price.max():,.0f} EUR")

**On the training rows, `price` is skewed 2.5512, and
`log1p(price)` is skewed -0.1320**: a long right tail becomes almost
symmetric. The training median is 14,490 EUR, the mean is 16,755, and the
maximum is 145,000, which is where the gap between median and mean comes
from.

This is measured on the training split rather than the whole file for the
same reason every feature comparison above was: a decision informed by rows
the evaluation will later hold out is a leak, and a target's own skew is no
exception, even though it carries no feature-target relationship to leak.

This is also a different decision from the ones above, and worth separating.
Transforming a **feature** changes the shape of an input. Transforming the
**target** changes what the model is optimising: on `log(price)` the model
minimises proportional error, so being 2,000 EUR wrong on a 10,000 EUR car counts
the same as being 20,000 EUR wrong on a 100,000 EUR one. That is often what a
price problem actually wants, and it also means the MAE that comes back is in log
units and cannot be compared with the euro figures in the table above without
transforming back.

The measurement is left here rather than acted on. The target's shape is a decision of its own.

Which columns would you transform on this file, and what would you
need to have measured before you could defend the answer?

In [ ]:
# Possible answer:
# mpg certainly, and then the honest answer is that the board cannot tell you
# about tax or mileage. mpg lowers the MAE by eight times its own SEM
# (-44.63 +/- 5.47); engineSize makes it clearly worse. tax at -2.89 +/- 0.83
# and mileage at +6.53 +/- 14.41 are three and seven euros on a 3,197 euro
# baseline, where changing the split alone moves 111.
#
# What has to be measured first: the baseline MAE with nothing transformed,
# and each candidate against it on the SAME splits, so the comparison is
# paired. Skew alone is not enough, and the table shows both ways it fails:
# tax lowers the error at a skew of -0.19, and engineSize raises it at 1.33,
# while year is flattened to a standard deviation of 0.001 and costs 0.61.
# And when the measurement cannot separate two candidates, say so and decide
# on something that does separate them, which here is the raw range each
# column carries into every later week.

# <font color='#E8800A'>The recipe, the log, and the seam</font> <a class="anchor" id="recipe"></a>
[Back to TOC](#toc)

Six questions were opened at the end of the exploration and six have
answers. What is left is to apply the rule once, on the file as it arrived, and
to record what each step did and why, including the step that was chosen against
the measurement.

<div class="alert alert-block alert-info">

**The recipe, in the order it must be applied.**

| # | step | what it changes |
|---|---|---|
| 1 | `Brand`, `transmission`, `fuelType` checked against their canon | 531 rows flagged |
| 2 | the eight numeric/target columns checked against their own domain | 86 rows flagged |
| 3 | the union of the two dropped, against repair, at a measured 12.82 EUR | -600 rows |
| 4 | missing values left open, **median / own level** carried forward | 1,537 cells |
| | **result** | 4,600 -> 4,000 rows, written to `cars4you.csv` |

The two checks overlap: 531 plus 86 is more than 600, because 17 rows fail
both. The rule does not care how many reasons a row has; one is already
enough. Neither check counts a missing value, which is why 1,537 of them come through
into the file.

**Row 4 is recorded, not applied.** A fill reads other rows to decide a cell's
value, so applying one here would compute it over rows that later become
someone's test set. The plan is written into the log instead, and each later
training split computes its own values from it.

**Three more decisions are recorded the same way**, because they belong to a
model rather than to the file: **`log1p` on `mpg` and `tax` and on nothing
else**, **no outlier treatment**, and **no scaler**. Every one of them was
measured on this problem rather than carried over from anywhere: a rule that
earns its place predicting a class does not carry to predicting a price, and
two of the three came out the opposite way from the plausible guess.

The log below records all seven and is written to
`logs/cars4you_cleaning_log.json`. **Each step carries its decision twice**:
once as the sentence you can read, and once as `carries`, a small structure a
later week can act on. A step described only in English can be read and not
applied, because the plan has to be worked out again from the sentence, and a
plan worked out again is a new decision wearing an old one's name.

</div>

__Step 18:__ Run the rule, and record the decision as you go.

In [ ]:
log = CleaningLog("cars4you")
log.record("Brand, transmission, fuelType",
           "drop rows whose value is not one of the canonical spellings",
           "a keystroke slip or a clipped word is not a category of its own, and "
           "there is no damaged cell here to repair back to -- these rows are, "
           "almost always, cars the spine draw never selected", int(text_bad.sum()),
           carries={"treatment": "drop", "canon": {c: sorted(v)
                                                   for c, v in canonical.items()}})
log.record("year, mileage, tax, mpg, engineSize, paintQuality%, previousOwners, price",
           "drop rows holding a value the column's own domain forbids; a missing value is "
           "left alone",
           "a negative distance or a fractional owner cannot be repaired without "
           "inventing the number, while an empty cell is unknown rather than "
           "wrong and is filled later, inside a training fold", int(num_bad.sum()),
           carries={"treatment": "drop", "missing_values": "left open"})
log.record("all 12 columns", "drop the union of the two rules above",
           "measured against repairing the spellings, dropping costs 12.82 EUR "
           "of mean absolute error, nearly five standard errors and 10 of 10 "
           "draws; it is recorded here because this file is indexed positionally "
           "by every later week, so its length is fixed, and the cost of that is "
           "a number rather than an assertion", int(bad.sum()),
           carries={"treatment": "drop", "cost_eur": 14.46, "rows_out": 4000,
                    "dtypes": {"year": "Int64", "previousOwners": "Int64"}})

# The fourth decision changes no cell, which is exactly why it is logged: a
# reader has to be able to see that leaving the missing values open was chosen,
# and a later week has to be able to read the plan that was chosen instead.
log.record("11 feature columns",
           "missing values LEFT OPEN; the 'median / own level' plan goes to later weeks",
           "a median computed here would be computed over the rows each later "
           "week holds out to score on, so the file would carry a value derived "
           "from its own test set; the plan won the comparison above and each "
           "week applies it inside its own split",
           int(candidate.isna().sum().sum()),
           carries={"applied": False,
                    "fill": {"numeric": "median", "categorical": "own level"},
                    "numeric": numeric, "categorical": categorical})

# Three more decisions belong to a model rather than to the file, so they are
# recorded and not applied either. Every one of them was measured on THIS
# problem: a rule that earns its place on a class does not carry to a price.
log.record("mpg, mileage", "log1p, and log1p on nothing else",
           "measured column by column against ten folds: mpg earns -44.11 EUR "
           "+/- 5.40 while engineSize costs +109.98 +/- 6.24 and year +384.87 "
           "+/- 16.12, and the plausible blanket rules lose, 'every skewed "
           "column' by +84.68 and 'all seven' by +79.95; tax at -2.79 +/- 0.87 "
           "and mileage at +7.85 +/- 14.43 are inside the 111 EUR the choice of "
           "split moves, so the score cannot separate them and the raw range "
           "does: mileage spans 150,000 against tax's 580, and a column that "
           "wide governs every distance and every first gradient step taken "
           "before scaling; taking mileage rather than tax costs 21.14 EUR here "
           "and is the same recipe in every week that inherits it",
           2,
           carries={"applied": False, "transform": "log1p",
                    "columns": ["mpg", "mileage"],
                    "rejected": ["engineSize", "year", "tax",
                                 "paintQuality%", "previousOwners"]})
log.record("7 numeric columns", "NO outlier treatment",
           "the IQR rule at k=1.5 removes 29.43% of the file, 1,043 rows on tax "
           "alone because 44.8% of cars sit at exactly 145 and the box is 20 "
           "euros wide; what it deletes is not a random third, the rows it "
           "keeps average 18,442 EUR against 12,617 for the rows it deletes, so "
           "it deletes the cheap end of the market the model will be asked about",
           0,
           carries={"applied": False, "outliers": "keep",
                    "rule_tested": "IQR k=1.5"})
log.record("the design matrix", "NO scaler",
           "measured over ten folds, the four candidates agree on MAE to within "
           "1.32 EUR against a standard error of about 40, because ordinary "
           "least squares on a full-rank design is scale-invariant in fit; "
           "cond(X) does fall from 4.312e+07 to 12.75, which is a fact about "
           "the solve and not about the score, and a model that is not "
           "scale-invariant has to measure this again for itself",
           0,
           carries={"applied": False, "scaler": None,
                    "reason": "OLS is scale-invariant in fit",
                    "conditioning": {"none": 4.312e+07, "standard": 12.75}})

print(pd.DataFrame(log.records()).drop(columns="reason").to_string(index=False))
print(f"\n{len(raw):,} rows in -> {int((~bad).sum()):,} rows out")

# Written, not only printed, and JSON rather than CSV: each step carries a
# machine-readable form of its decision beside the prose, and a CSV would
# flatten that back into text.
log.to_json("../../logs/cars4you_cleaning_log.json")
print(f"log written: {len(log)} decisions,"
      f" {sum(step.carries is not None for step in log.steps)} carrying a plan")

`year` and `previousOwners` still read as `float64` here.
Nothing about a registration year or an owner count is fractional; the column
only carries a float dtype because some of the 600 dropped rows held a
fractional or invalid value, and one bad cell forces the whole column's dtype
at read time.

Once those rows are gone the column holds whole numbers, and it still holds
missing values, so `int` cannot express it: `astype(int)` raises the moment it meets a
missing value. `Int64`, with a capital I, is pandas' nullable integer, and it
says both things at once. It is also the dtype every later week names when it
opens this file, which is why a CSV that carries no dtypes of its own does not
lose the distinction.

__Step 19:__ The seam: does the rule reproduce cars4you.csv, in its own order?

In [ ]:
# `drop_flagged` is the fix the rule above was given, so the file comes out
# of the same function that produced the count: change the fix, and the file
# changes with it rather than staying behind.
cars = drop_flagged(raw, bad).sort_index().reset_index(drop=True)

# Int64, not int. Both columns hold whole numbers AND missing values, and plain `int`
# cannot hold a missing value at all: it raises rather than rounding or blanking. Int64
# is pandas' nullable integer, so one dtype can say "a whole number, and this
# one is missing".
DTYPES = {"year": "Int64", "previousOwners": "Int64"}
cars = cars.astype(DTYPES)

cars.to_csv("../../data/interim/cars4you.csv", index=False)

print(f"written: {cars.shape[0]:,} rows x {cars.shape[1]} columns,"
      f" {cars.isna().sum().sum():,} missing values")

<div class="alert alert-block alert-success">

**4,600 rows became 4,000.** `cars4you.csv` is not handed
down from anywhere: it is the output of the rule above, applied to
`cars4you_raw.csv`.

</div>

# <font color='#E8800A'>Key takeaways</font> <a class="anchor" id="takeaways"></a>
[Back to TOC](#toc)

What this session established:

1. **The file you model on is one you produced, and the rule that produced
   it is a decision.** 600 of 4,600 rows failed their own column's rule and
   left; 514 of them for a misspelt label and nothing more. The rule was written
   from what each column measures, before any statistic was computed, and the
   target was checked by the same rule as everything else.
2. **A treatment is not good or bad; it fits a file or it does not.** Repairing
   the spellings beat dropping the row by 14.46 EUR, nearly six standard errors,
   on 10 of 10 draws. Blanking came within 2.01 +/- 2.43 of dropping, which is
   no difference at all. When a constraint from outside the data overrode the
   winner,
   having measured is what let the cost be named: under half of one percent.
3. **A missing value is a decision, not a defect.** No rule here flags a row for being
   incomplete, so 1,537 empty cells survive into the finished file, and what
   they become is measured like everything else: the training median and a level
   of missingness' own, against a mean, a zero and the training mode. Only two
   candidates moved the score, and zero moved it by 639 EUR.
4. **Where a fill is computed matters more than which fill it is.** Every value
   here is computed on the training rows of the fold it is used in. A median
   taken over the whole file would have been taken partly from the cars the
   model is about to be scored on.
5. **Cardinality is a structural cost you can measure before it hurts.** Four
   label columns encode to 326 columns, or 18 without `model`. That number is
   available before any model is fitted.
6. **An outlier rule is a screening device.** The 1.5 x IQR fence removed 29.43%
   of the file, flagged 1,043 rows on `tax` alone because nearly half the cars
   share one value, and deleted the cheap end of the market. The rows it called
   anomalous on `mpg` turned out to be hybrids.
7. **A transform is selected per column, on held-out rows.** `mpg` earns it,
   `engineSize` and `year` do not, and the rule "log1p everything skewed" is 84.68
   EUR worse than leaving the file alone.
8. **A mean without its uncertainty is not a result.** +7.85 +/- 14.43 and
   +109.98 +/- 6.24 are different kinds of number, and one split cannot tell them
   apart.

Exit ticket: `tax` has a skew of -0.19 and its transform earns its
place; `engineSize` has a skew of 1.33 and its transform is rejected. What does
that pair of results tell you about selecting transforms by skew?

In [ ]:
# Possible answer:
# That skew is a symptom, not the criterion. It says a column has a tail; it
# does not say the tail is hurting this model. engineSize is skewed and its
# values span one order of magnitude, so log1p compresses nothing and bends a
# relationship that was already linear. tax is barely skewed and the transform
# still helps. The only thing that settles it is a paired comparison on rows
# the model did not train on.

# <font color='#E8800A'>References</font> <a class="anchor" id="references"></a>
[Back to TOC](#toc)

- pandas, [`DataFrame.describe`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.describe.html) · [`DataFrame.skew`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.skew.html)
- NumPy, [`log1p`](https://numpy.org/doc/stable/reference/generated/numpy.log1p.html)
- scikit-learn, [`OneHotEncoder`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html) · [`mean_absolute_error`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.mean_absolute_error.html)